##### 04 - Data Quality Framework

A reusable, configurable data quality engine that:
1. Defines validation rules per table (nullability, range, regex, referential integrity)
2. Runs checks and computes a quality score
3. Logs results to an audit Delta table
4. Quarantines failed records for investigation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from datetime import datetime
from typing import List, Dict, Any

SILVER_SCHEMA = "ecommerce_silver"
AUDIT_SCHEMA = "ecommerce_audit"


##### 1. Define the Quality Rule Engine

In [0]:
class DataQualityRule:
    """Represents a single data quality check."""

    def __init__(self, rule_name: str, rule_type: str, column: str,
                 condition: str = None, ref_table: str = None, severity: str = "error"):
        self.rule_name = rule_name
        self.rule_type = rule_type
        self.column = column
        self.condition = condition
        self.ref_table = ref_table
        self.severity = severity


class DataQualityEngine:
    """Runs DQ checks and produces metrics + quarantined records."""

    def __init__(self, spark_session, table_name: str, df: DataFrame, rules: List[DataQualityRule]):
        self.spark = spark_session
        self.table_name = table_name
        self.df = df
        self.rules = rules
        self.results = []
        self.run_ts = datetime.now()

    def _check_not_null(self, rule: DataQualityRule) -> int:
        return self.df.filter(F.col(rule.column).isNull()).count()

    def _check_unique(self, rule: DataQualityRule) -> int:
        total = self.df.count()
        distinct = self.df.select(rule.column).distinct().count()
        return total - distinct

    def _check_range(self, rule: DataQualityRule) -> int:
        return self.df.filter(~F.expr(rule.condition)).count()

    def _check_regex(self, rule: DataQualityRule) -> int:
        return self.df.filter(
            F.col(rule.column).isNotNull() & ~F.col(rule.column).rlike(rule.condition)
        ).count()

    def _check_referential(self, rule: DataQualityRule) -> int:
        ref_df = self.spark.table(rule.ref_table)
        orphans = self.df.join(ref_df, self.df[rule.column] == ref_df[rule.column], "left_anti")
        return orphans.count()

    def _check_custom(self, rule: DataQualityRule) -> int:
        return self.df.filter(~F.expr(rule.condition)).count()

    def run(self) -> List[Dict[str, Any]]:
        total_rows = self.df.count()

        check_dispatch = {
            "not_null": self._check_not_null,
            "unique": self._check_unique,
            "range": self._check_range,
            "regex": self._check_regex,
            "referential": self._check_referential,
            "custom": self._check_custom,
        }

        for rule in self.rules:
            checker = check_dispatch.get(rule.rule_type)
            if not checker:
                continue

            failed_count = checker(rule)
            passed_count = total_rows - failed_count
            pass_rate = round((passed_count / total_rows) * 100, 2) if total_rows > 0 else 0

            self.results.append({
                "table_name": self.table_name,
                "rule_name": rule.rule_name,
                "rule_type": rule.rule_type,
                "column": rule.column,
                "severity": rule.severity,
                "total_rows": total_rows,
                "passed": passed_count,
                "failed": failed_count,
                "pass_rate_pct": pass_rate,
                "run_timestamp": str(self.run_ts),
            })

        return self.results

    def get_quality_score(self) -> float:
        if not self.results:
            return 0.0
        error_results = [r for r in self.results if r["severity"] == "error"]
        if not error_results:
            return 100.0
        return round(sum(r["pass_rate_pct"] for r in error_results) / len(error_results), 2)

    def get_quarantine_df(self) -> DataFrame:
        quarantine_conditions = []
        for rule in self.rules:
            if rule.severity != "error":
                continue
            if rule.rule_type == "not_null":
                quarantine_conditions.append(F.col(rule.column).isNull())
            elif rule.rule_type in ("range", "custom"):
                quarantine_conditions.append(~F.expr(rule.condition))

        if not quarantine_conditions:
            return self.spark.createDataFrame([], self.df.schema)

        combined = quarantine_conditions[0]
        for cond in quarantine_conditions[1:]:
            combined = combined | cond

        return self.df.filter(combined)


##### 2. Define Rules for Each Silver Table

In [0]:
customer_rules = [
    DataQualityRule("customer_id_not_null", "not_null", "customer_id", severity="error"),
    DataQualityRule("customer_id_unique", "unique", "customer_id", severity="error"),
    DataQualityRule("email_format", "regex", "email",
                    condition=r"^[\w\.\-]+@[\w\.\-]+\.\w+$", severity="warning"),
    DataQualityRule("age_range", "range", "age",
                    condition="age BETWEEN 13 AND 120", severity="warning"),
    DataQualityRule("signup_date_not_null", "not_null", "signup_date", severity="error"),
    DataQualityRule("valid_status", "custom", "status",
                    condition="status IN ('active', 'inactive')", severity="error"),
]

order_rules = [
    DataQualityRule("order_id_not_null", "not_null", "order_id", severity="error"),
    DataQualityRule("order_id_unique", "unique", "order_id", severity="error"),
    DataQualityRule("total_positive", "range", "total_amount",
                    condition="total_amount >= 0", severity="error"),
    DataQualityRule("valid_status", "custom", "status",
                    condition="status IN ('completed','shipped','processing','cancelled','returned')",
                    severity="error"),
    DataQualityRule("customer_ref", "referential", "customer_id",
                    ref_table=f"{SILVER_SCHEMA}.customers", severity="error"),
    DataQualityRule("order_date_not_null", "not_null", "order_date", severity="error"),
]

product_rules = [
    DataQualityRule("product_id_not_null", "not_null", "product_id", severity="error"),
    DataQualityRule("product_id_unique", "unique", "product_id", severity="error"),
    DataQualityRule("price_positive", "range", "price", condition="price > 0", severity="error"),
    DataQualityRule("rating_range", "range", "rating",
                    condition="rating BETWEEN 0 AND 5 OR rating IS NULL", severity="warning"),
]

item_rules = [
    DataQualityRule("item_id_not_null", "not_null", "item_id", severity="error"),
    DataQualityRule("quantity_positive", "range", "quantity",
                    condition="quantity > 0", severity="error"),
    DataQualityRule("unit_price_positive", "range", "unit_price",
                    condition="unit_price > 0", severity="error"),
    DataQualityRule("order_ref", "referential", "order_id",
                    ref_table=f"{SILVER_SCHEMA}.orders", severity="error"),
    DataQualityRule("product_ref", "referential", "product_id",
                    ref_table=f"{SILVER_SCHEMA}.products", severity="error"),
]

##### 3. Run Quality Checks

In [0]:
all_results = []
quality_scores = {}

table_configs = [
    ("silver_customers", f"{SILVER_SCHEMA}.customers", customer_rules),
    ("silver_orders", f"{SILVER_SCHEMA}.orders", order_rules),
    ("silver_products", f"{SILVER_SCHEMA}.products", product_rules),
    ("silver_order_items", f"{SILVER_SCHEMA}.order_items", item_rules),
]

for table_name, source_table, rules in table_configs:
    print(f"\n{'='*60}")
    print(f"Running DQ checks on: {table_name}")
    print(f"{'='*60}")

    df = spark.table(source_table)
    engine = DataQualityEngine(spark, table_name, df, rules)
    results = engine.run()
    score = engine.get_quality_score()
    quality_scores[table_name] = score

    all_results.extend(results)

    for r in results:
        status = "PASS" if r["pass_rate_pct"] == 100 else "WARN" if r["severity"] == "warning" else "FAIL"
        print(f"  [{status}] {r['rule_name']}: {r['pass_rate_pct']}% ({r['failed']} failures)")

    print(f"\n  Quality Score: {score}%")



Running DQ checks on: silver_customers
  [PASS] customer_id_not_null: 100.0% (0 failures)
  [PASS] customer_id_unique: 100.0% (0 failures)
  [PASS] email_format: 100.0% (0 failures)
  [PASS] age_range: 100.0% (0 failures)
  [PASS] signup_date_not_null: 100.0% (0 failures)
  [PASS] valid_status: 100.0% (0 failures)

  Quality Score: 100.0%

Running DQ checks on: silver_orders
  [PASS] order_id_not_null: 100.0% (0 failures)
  [PASS] order_id_unique: 100.0% (0 failures)
  [PASS] total_positive: 100.0% (0 failures)
  [PASS] valid_status: 100.0% (0 failures)
  [PASS] customer_ref: 100.0% (0 failures)
  [PASS] order_date_not_null: 100.0% (0 failures)

  Quality Score: 100.0%

Running DQ checks on: silver_products
  [PASS] product_id_not_null: 100.0% (0 failures)
  [PASS] product_id_unique: 100.0% (0 failures)
  [PASS] price_positive: 100.0% (0 failures)
  [PASS] rating_range: 100.0% (0 failures)

  Quality Score: 100.0%

Running DQ checks on: silver_order_items
  [PASS] item_id_not_null: 10

##### 4. Write Audit Log

In [0]:
df_audit = spark.createDataFrame(all_results)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{AUDIT_SCHEMA}.dq_audit_log")
)

print("Audit log written.")
print(f"\nTotal rules evaluated: {len(all_results)}")
print(f"Rules passed (100%): {sum(1 for r in all_results if r['pass_rate_pct'] == 100)}")
print(f"Rules with issues:   {sum(1 for r in all_results if r['pass_rate_pct'] < 100)}")


Audit log written.

Total rules evaluated: 21
Rules passed (100%): 21
Rules with issues:   0


##### 5. Quarantine Failed Records

In [0]:
df_orders = spark.table(f"{SILVER_SCHEMA}.orders")
engine = DataQualityEngine(spark, "silver_orders", df_orders, order_rules)
engine.run()

df_quarantine = engine.get_quarantine_df()

if df_quarantine.count() > 0:
    (
        df_quarantine
        .withColumn("_quarantine_reason", F.lit("DQ check failure"))
        .withColumn("_quarantine_timestamp", F.current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{AUDIT_SCHEMA}.quarantine_orders")
    )
    print(f"Quarantined {df_quarantine.count()} order records")
else:
    print("No records quarantined - all checks passed!")


No records quarantined - all checks passed!


##### 6. Quality Dashboard View

In [0]:
print("\n" + "="*60)
print("        DATA QUALITY SCORECARD")
print("="*60)
for table, score in quality_scores.items():
    bar_len = int(score / 5)
    bar = "#" * bar_len + "-" * (20 - bar_len)
    grade = "A" if score >= 95 else "B" if score >= 85 else "C" if score >= 70 else "D"
    print(f"  {table:<25} [{bar}] {score:>6.2f}%  Grade: {grade}")
print("="*60)



        DATA QUALITY SCORECARD
  silver_customers          [####################] 100.00%  Grade: A
  silver_orders             [####################] 100.00%  Grade: A
  silver_products           [####################] 100.00%  Grade: A
  silver_order_items        [####################] 100.00%  Grade: A


##### 7. Historical Quality Trend (via Delta Time Travel)

In [0]:
%sql
SELECT
  run_timestamp,
  table_name,
  ROUND(AVG(pass_rate_pct), 2) AS avg_pass_rate,
  COUNT(*) AS rules_checked,
  SUM(CASE WHEN pass_rate_pct < 100 THEN 1 ELSE 0 END) AS rules_failed
FROM ecommerce_audit.dq_audit_log
GROUP BY run_timestamp, table_name
ORDER BY run_timestamp DESC, table_name


run_timestamp,table_name,avg_pass_rate,rules_checked,rules_failed
2026-05-01 12:39:06.908457,silver_order_items,100.0,5,0
2026-05-01 12:39:03.196875,silver_products,100.0,4,0
2026-05-01 12:38:57.213750,silver_orders,100.0,6,0
2026-05-01 12:38:14.630723,silver_customers,100.0,6,0


In [0]:
print("Data quality checks complete! Proceed to notebook 05_incremental_processing.")

Data quality checks complete! Proceed to notebook 05_incremental_processing.
